In [0]:
# Load Silver tables
print("=== LOADING SILVER TABLES ===\n")

customers = spark.table("food_delivery.silver.customers")
restaurants = spark.table("food_delivery.silver.restaurants")
drivers = spark.table("food_delivery.silver.drivers")
orders = spark.table("food_delivery.silver.orders")
order_items = spark.table("food_delivery.silver.order_items")
payments = spark.table("food_delivery.silver.payments")

print("✅ All Silver tables loaded successfully!")

=== LOADING SILVER TABLES ===

✅ All Silver tables loaded successfully!


In [0]:
from pyspark.sql.functions import sum, count, avg, round as spark_round, col

print("=== CREATING REVENUE VIEWS ===\n")

# 1. Revenue by City
revenue_by_city = orders.join(restaurants, "restaurant_id") \
    .groupBy("city") \
    .agg(
        count("order_id").alias("total_orders"),
        sum("total_amount").alias("total_revenue"),
        avg("total_amount").alias("avg_order_value")
    ) \
    .withColumn("total_revenue", spark_round(col("total_revenue"), 2)) \
    .withColumn("avg_order_value", spark_round(col("avg_order_value"), 2))

print("✅ Revenue by City created")

# 2. Revenue by Cuisine
revenue_by_cuisine = orders.join(restaurants, "restaurant_id") \
    .groupBy("cuisine") \
    .agg(
        count("order_id").alias("total_orders"),
        sum("total_amount").alias("total_revenue"),
        avg("total_amount").alias("avg_order_value")
    ) \
    .withColumn("total_revenue", spark_round(col("total_revenue"), 2)) \
    .withColumn("avg_order_value", spark_round(col("avg_order_value"), 2))

print("✅ Revenue by Cuisine created")

# 3. Daily Revenue Trend
daily_revenue = orders \
    .groupBy("order_date") \
    .agg(
        count("order_id").alias("daily_orders"),
        sum("total_amount").alias("daily_revenue"),
        avg("total_amount").alias("avg_order_value")
    ) \
    .withColumn("daily_revenue", spark_round(col("daily_revenue"), 2)) \
    .withColumn("avg_order_value", spark_round(col("avg_order_value"), 2)) \
    .orderBy("order_date")

print("✅ Daily Revenue Trend created")

=== CREATING REVENUE VIEWS ===

✅ Revenue by City created
✅ Revenue by Cuisine created
✅ Daily Revenue Trend created


In [0]:
print("=== CREATING CUSTOMER ANALYTICS ===\n")

# 1. Customer Lifetime Value (CLV)
customer_clv = customers.join(orders, "customer_id") \
    .groupBy("customer_id", "name", "city") \
    .agg(
        count("order_id").alias("total_orders"),
        sum("total_amount").alias("total_spent"),
        avg("total_amount").alias("avg_order_value")
    ) \
    .withColumn("total_spent", spark_round(col("total_spent"), 2)) \
    .withColumn("avg_order_value", spark_round(col("avg_order_value"), 2)) \
    .orderBy(col("total_spent").desc())

print("✅ Customer Lifetime Value created")

# 2. Top Customers
top_customers = customer_clv.limit(10)
print("✅ Top 10 Customers identified")

# 3. Customer City Distribution
customer_city_dist = customers.groupBy("city") \
    .agg(count("customer_id").alias("customer_count")) \
    .orderBy(col("customer_count").desc())

print("✅ Customer City Distribution created")

=== CREATING CUSTOMER ANALYTICS ===

✅ Customer Lifetime Value created
✅ Top 10 Customers identified
✅ Customer City Distribution created


In [0]:
print("=== CREATING DRIVER PERFORMANCE ===\n")

# 1. Driver Performance Metrics
driver_performance = drivers.join(orders, "driver_id") \
    .groupBy("driver_id", "driver_name", "vehicle") \
    .agg(
        count("order_id").alias("total_deliveries"),
        sum("total_amount").alias("total_revenue_generated"),
        avg("total_amount").alias("avg_order_value")
    ) \
    .withColumn("total_revenue_generated", spark_round(col("total_revenue_generated"), 2)) \
    .withColumn("avg_order_value", spark_round(col("avg_order_value"), 2)) \
    .orderBy(col("total_deliveries").desc())

print("✅ Driver Performance created")

# 2. Top Drivers
top_drivers = driver_performance.limit(10)
print("✅ Top 10 Drivers identified")

=== CREATING DRIVER PERFORMANCE ===

✅ Driver Performance created
✅ Top 10 Drivers identified


In [0]:
print("=== CREATING TIME-BASED ANALYTICS ===\n")

# 1. Orders by Hour
orders_by_hour = orders.groupBy("order_hour") \
    .agg(
        count("order_id").alias("order_count"),
        sum("total_amount").alias("revenue")
    ) \
    .withColumn("revenue", spark_round(col("revenue"), 2)) \
    .orderBy("order_hour")

print("✅ Orders by Hour created")

# 2. Orders by Day of Week
orders_by_day = orders.groupBy("order_dayofweek") \
    .agg(
        count("order_id").alias("order_count"),
        sum("total_amount").alias("revenue")
    ) \
    .withColumn("revenue", spark_round(col("revenue"), 2)) \
    .orderBy("order_dayofweek")

print("✅ Orders by Day created")

# 3. Weekend vs Weekday Analysis
weekend_analysis = orders.groupBy("is_weekend") \
    .agg(
        count("order_id").alias("order_count"),
        sum("total_amount").alias("revenue"),
        avg("total_amount").alias("avg_order_value")
    ) \
    .withColumn("revenue", spark_round(col("revenue"), 2)) \
    .withColumn("avg_order_value", spark_round(col("avg_order_value"), 2))

print("✅ Weekend vs Weekday Analysis created")

=== CREATING TIME-BASED ANALYTICS ===

✅ Orders by Hour created
✅ Orders by Day created
✅ Weekend vs Weekday Analysis created


In [0]:
print("=== SAVING GOLD TABLES ===\n")

# Save revenue views
revenue_by_city.write.format("delta").mode("overwrite").saveAsTable("food_delivery.gold.revenue_by_city")
revenue_by_cuisine.write.format("delta").mode("overwrite").saveAsTable("food_delivery.gold.revenue_by_cuisine")
daily_revenue.write.format("delta").mode("overwrite").saveAsTable("food_delivery.gold.daily_revenue")

# Save customer analytics
customer_clv.write.format("delta").mode("overwrite").saveAsTable("food_delivery.gold.customer_clv")
top_customers.write.format("delta").mode("overwrite").saveAsTable("food_delivery.gold.top_customers")
customer_city_dist.write.format("delta").mode("overwrite").saveAsTable("food_delivery.gold.customer_city_dist")

# Save driver performance
driver_performance.write.format("delta").mode("overwrite").saveAsTable("food_delivery.gold.driver_performance")
top_drivers.write.format("delta").mode("overwrite").saveAsTable("food_delivery.gold.top_drivers")

# Save time-based analytics
orders_by_hour.write.format("delta").mode("overwrite").saveAsTable("food_delivery.gold.orders_by_hour")
orders_by_day.write.format("delta").mode("overwrite").saveAsTable("food_delivery.gold.orders_by_day")
weekend_analysis.write.format("delta").mode("overwrite").saveAsTable("food_delivery.gold.weekend_analysis")

print("✅ All Gold tables saved successfully!")
print("📊 Gold tables created in: food_delivery.gold.*")

=== SAVING GOLD TABLES ===

✅ All Gold tables saved successfully!
📊 Gold tables created in: food_delivery.gold.*


In [0]:
print("=== GOLD TABLE VERIFICATION ===\n")

# List all Gold tables
print("📋 All Gold Tables:")
display(spark.sql("SHOW TABLES IN food_delivery.gold"))

# Preview each table
tables_to_view = [
    ("revenue_by_city", "Revenue by City"),
    ("revenue_by_cuisine", "Revenue by Cuisine"),
    ("daily_revenue", "Daily Revenue Trend"),
    ("customer_clv", "Customer Lifetime Value"),
    ("top_customers", "Top 10 Customers"),
    ("driver_performance", "Driver Performance"),
    ("orders_by_hour", "Orders by Hour")
]

for table_name, display_name in tables_to_view:
    print(f"\n📊 {display_name}:")
    display(spark.sql(f"SELECT * FROM food_delivery.gold.{table_name} LIMIT 10"))

=== GOLD TABLE VERIFICATION ===

📋 All Gold Tables:


database,tableName,isTemporary
gold,customer_city_dist,false
gold,customer_clv,false
gold,daily_revenue,false
gold,driver_performance,false
gold,orders_by_day,false
gold,orders_by_hour,false
gold,revenue_by_city,false
gold,revenue_by_cuisine,false
gold,top_customers,false
gold,top_drivers,false



📊 Revenue by City:


city,total_orders,total_revenue,avg_order_value
Dubai,1800,340729.75,189.29
Al Ain,1186,223771.05,188.68
Sharjah,2882,549822.75,190.78
Ajman,2446,462199.10,188.96
Abu Dhabi,1686,310999.00,184.46



📊 Revenue by Cuisine:


...

customer_id,name,city,total_orders,total_spent,avg_order_value
992,Customer 992,Al Ain,17,4347.20,255.72
15,Customer 15,Abu Dhabi,20,4325.60,216.28
554,Customer 554,Ajman,15,4242.75,282.85
694,Customer 694,Ajman,18,4074.70,226.37
37,Customer 37,Dubai,18,4008.80,222.71
145,Customer 145,Sharjah,19,3946.65,207.72
109,Customer 109,Sharjah,20,3907.85,195.39
794,Customer 794,Dubai,16,3896.25,243.52
423,Customer 423,Abu Dhabi,18,3863.10,214.62
232,Customer 232,Ajman,17,3851.60,226.56



📊 Driver Performance:


driver_id,driver_name,vehicle,total_deliveries,total_revenue_generated,avg_order_value
158,Driver 158,Bike,67,12620.75,188.37
121,Driver 121,Car,67,11857.15,176.97
118,Driver 118,Bike,65,13074.60,201.15
96,Driver 96,Bike,65,10862.35,167.11
51,Driver 51,Car,65,11915.80,183.32
178,Driver 178,Bike,64,12170.95,190.17
148,Driver 148,Car,62,10803.70,174.25
177,Driver 177,Car,62,11206.80,180.75
53,Driver 53,Car,62,9906.30,159.78
153,Driver 153,Car,62,13312.85,214.72



📊 Orders by Hour:


order_hour,order_count,revenue
0,428,78384.45
1,405,72664.40
2,372,70045.10
3,406,74982.85
4,423,81239.50
5,426,78966.60
6,402,76837.40
7,388,73824.00
8,405,75117.50
9,399,74457.00


In [0]:
print("=== EXECUTIVE DASHBOARD VIEW ===\n")

# Create a comprehensive executive summary
executive_summary = spark.sql("""
    SELECT 
        (SELECT COUNT(*) FROM food_delivery.silver.customers) as total_customers,
        (SELECT COUNT(*) FROM food_delivery.silver.restaurants) as total_restaurants,
        (SELECT COUNT(*) FROM food_delivery.silver.drivers) as total_drivers,
        (SELECT COUNT(*) FROM food_delivery.silver.orders) as total_orders,
        (SELECT SUM(total_amount) FROM food_delivery.silver.orders) as total_revenue,
        (SELECT AVG(total_amount) FROM food_delivery.silver.orders) as avg_order_value,
        (SELECT COUNT(DISTINCT order_date) FROM food_delivery.silver.orders) as active_days
""")

# Save executive summary
executive_summary.write.format("delta").mode("overwrite").saveAsTable("food_delivery.gold.executive_summary")

print("✅ Executive Summary created!")

# Display it
display(spark.sql("SELECT * FROM food_delivery.gold.executive_summary"))

=== EXECUTIVE DASHBOARD VIEW ===

✅ Executive Summary created!


total_customers,total_restaurants,total_drivers,total_orders,total_revenue,avg_order_value,active_days
1000,100,200,10000,1887521.65,188.752165,365


In [0]:
print("=" * 70)
print("🎉 GOLD LAYER COMPLETE! FOOD DELIVERY DATA PIPELINE IS READY!")
print("=" * 70)

print("\n📊 Complete Architecture Summary:")
print("-" * 50)
print("📁 Bronze Layer (Raw Data):")
print("   • 6 tables with cleaned duplicates")
print("   • Stored as Delta tables")
print("\n📁 Silver Layer (Cleaned & Enriched):")
print("   • Fixed data types")
print("   • Handled NULL values")
print("   • Added derived columns")
print("   • 6 enriched tables")
print("\n📁 Gold Layer (Business Metrics):")
print("   • Revenue analytics by city & cuisine")
print("   • Customer lifetime value")
print("   • Driver performance metrics")
print("   • Time-based analytics")
print("   • Executive dashboard")
print("   • 11 business-ready views/tables")

print("\n📊 Key Business Metrics:")
print("-" * 50)
display(spark.sql("SELECT * FROM food_delivery.gold.executive_summary"))

print("\n🏆 Top Performing Cities:")
display(spark.sql("SELECT * FROM food_delivery.gold.revenue_by_city LIMIT 5"))

print("\n🍽️ Popular Cuisines:")
display(spark.sql("SELECT * FROM food_delivery.gold.revenue_by_cuisine LIMIT 5"))

print("\n" + "=" * 70)
print("✅ YOUR FOOD DELIVERY DATA PIPELINE IS COMPLETE!")
print("=" * 70)
print("\n🎯 What you've built:")
print("   ✅ End-to-end ETL pipeline (Bronze → Silver → Gold)")
print("   ✅ Production-ready Delta Lake architecture")
print("   ✅ Business intelligence views")
print("   ✅ Ready for Power BI visualization")
print("\n📝 This is a portfolio-quality project!")
print("   • Databricks certified skills")
print("   • PySpark & Delta Lake expertise")
print("   • Data engineering best practices")
print("   • Business analytics capabilities")

🎉 GOLD LAYER COMPLETE! FOOD DELIVERY DATA PIPELINE IS READY!

📊 Complete Architecture Summary:
--------------------------------------------------
📁 Bronze Layer (Raw Data):
   • 6 tables with cleaned duplicates
   • Stored as Delta tables

📁 Silver Layer (Cleaned & Enriched):
   • Fixed data types
   • Handled NULL values
   • Added derived columns
   • 6 enriched tables

📁 Gold Layer (Business Metrics):
   • Revenue analytics by city & cuisine
   • Customer lifetime value
   • Driver performance metrics
   • Time-based analytics
   • Executive dashboard
   • 11 business-ready views/tables

📊 Key Business Metrics:
--------------------------------------------------


total_customers,total_restaurants,total_drivers,total_orders,total_revenue,avg_order_value,active_days
1000,100,200,10000,1887521.65,188.752165,365



🏆 Top Performing Cities:


city,total_orders,total_revenue,avg_order_value
Dubai,1800,340729.75,189.29
Al Ain,1186,223771.05,188.68
Sharjah,2882,549822.75,190.78
Ajman,2446,462199.10,188.96
Abu Dhabi,1686,310999.00,184.46



🍽️ Popular Cuisines:


cuisine,total_orders,total_revenue,avg_order_value
Italian,1304,250443.45,192.06
Indian,1291,238993.75,185.12
Pizza,2070,388874.70,187.86
Chinese,1724,318851.45,184.95
Arabic,1620,309135.80,190.82



✅ YOUR FOOD DELIVERY DATA PIPELINE IS COMPLETE!

🎯 What you've built:
   ✅ End-to-end ETL pipeline (Bronze → Silver → Gold)
   ✅ Production-ready Delta Lake architecture
   ✅ Business intelligence views
   ✅ Ready for Power BI visualization

📝 This is a portfolio-quality project!
   • Databricks certified skills
   • PySpark & Delta Lake expertise
   • Data engineering best practices
   • Business analytics capabilities
